# AIIJC 2026 · AIC — финальное решение `disentangle_b2_li760_r8_all_data_hard_pixel_ft`

Задача: по JPEG-изображению предсказать бинарную маску области, изменённой ИИ.
Метрика — **AIC Score**, гармоническое среднее `Dice_pos` (по позитивным примерам)
и `(1 - FPR_neg)` (по негативным). Жёсткие ограничения соревнования: **≤ 100 GFLOPs**
на изображение (`torch.utils.flop_counter.FlopCounterMode`) и **≤ 50 мс** на одном
H100 80 GB. Подробности — `AIIJC_RULES.md`.

Этот ноутбук — единственный источник правды для воспроизведения финального решения:
установка зависимостей → подготовка данных → три стадии обучения по порядку →
проверка бюджета GFLOPs → инференс и сборка `submission.csv` + `predictions/`.

Разбор данных (EDA), сравнение архитектур по `runs/` и абляции вынесены в
отдельные ноутбуки: `notebooks/eda.ipynb`, `notebooks/model_comparison.ipynb`,
`notebooks/ablations.ipynb` — они объясняют, **почему** решение выглядит именно
так, этот ноутбук — только **как** его воспроизвести.

## Архитектура одним абзацем

`PVT-v2-B2-li` (timm-энкодер с линейным вниманием) читает RGB-кадр; параллельно
нативные JPEG DCT-коэффициенты проходят через forensic-ветку (`ForensicFusion`)
и сливаются с признаками энкодера на страйдах 8/16/32. Дальше признаки проходят
через `ForensicDisentangle` (DG-Force: PFD/EFD бутылочные горлышки на страйдах
4/8/16/32 + cross-attention 16→32 на страйде 32, канальный гейт с нулевой
инициализацией) и декодируются `EMCADDecoder` в маску, плюс отдельная
классификационная голова (`GateHead`) — предсказывает, изменено ли изображение
вообще. Подробнее: `configs/experiments/disentangle.md`.

## Расшифровка имени run (проверено по конфигам, не по памяти)

| Токен | Значение | Где заведено |
|---|---|---|
| `b2` | энкодер семейства PVTv2-B2 | `model.encoder` |
| `li` | суффикс `_li` timm-энкодера `pvt_v2_b2_li` (вариант с линейным вниманием) | `configs/experiments/disentangle_b2_li760_long.yaml` |
| `760` | `dataset.image_size = 760` (RGB вход) — максимальный размер с шагом 8 при native JPEG до Full HD, укладывающийся в 100 GFLOPs | `disentangle_b2_li760_long.yaml` |
| `r8` | `model.disentangle_reduction = 8` (канальное горлышко PFD/EFD C → C/8) | `disentangle_b2_li760_r8_long.yaml` |
| `all_data` | `train.train_all_data = true` — финальный дотюн на train + development + holdout | `disentangle_b2_li760_r8_all_data_ft.yaml` |
| `hard_pixel` | `loss.hard_pixel_weight = 0.1` — доп. BCE по 10% самых сложных пикселей внутри/вне GT | `disentangle_b2_li760_r8_all_data_hard_pixel_ft.yaml` |
| `ft` | дотюн (`finetune_from`) от EMA-весов предыдущей стадии, новый optimizer/EMA | во всех `*_ft.yaml` |

`resize_mode` в имени **не имеет отношения к letterbox** — в текущем пайплайне
(`pipeline_version: jpeg640_v1`) такого поля в `DatasetConfig` вообще нет
(letterbox — поле только legacy-адаптера `emcad_v1`, отклоняется как неизвестный
ключ). Это перепроверено по `src/config.py`, а не взято из внешних заметок.


**TODO (авторам, на проверку):** ниже — черновая формулировка того, почему выбран
именно этот финальный конфиг, а не более ранние (`disentangle_b2_li760_r8_long`
без all-data дотюна, или `..._all_data_ft` без hard-pixel лосса). Проверьте цифры
и, если не согласны с интерпретацией — перепишите:

- `disentangle_b2_li760_r8_long` — базовое обучение (18 эпох) даёт модель, которая
  никогда не видела development/holdout; это осознанный выбор ради независимой
  выборки для подбора чекпоинта и порогов.
- `..._all_data_ft` дообучает 3 полных прохода на объединении train+development+
  holdout (после того как чекпоинт и пороги уже зафиксированы на предыдущей
  стадии) — это стандартный приём «дотюн на всех размеченных данных перед
  сдачей», а не переобучение на валидации внутри одной оценки.
- `..._all_data_hard_pixel_ft` добавляет к этому ещё 3 прохода с hard-pixel
  лоссом, ужесточающим градиент на границе маски. Валидации на этой стадии нет
  (см. markdown исходных ноутбуков стадий) — это финальный «слепой» дотюн,
  поэтому фактический прирост AIC от hard-pixel компоненты не измерен отдельным
  контролируемым прогоном (сравнение с/без — см. `notebooks/ablations.ipynb`,
  раздел «hard-pixel loss»; там честно указано, что это не A/B, а только анализ
  по логам loss).


## 1. Установка зависимостей

Проект использует conda-окружение `challenges` (`environment.yml`). Чтобы решение
запускалось «с нуля» без ручных шагов, ставим тот же набор пакетов прямо в
ноутбуке через `pip`. `triton` доступен под Linux, `triton-windows` — под Windows
(используется CUDA-веткой поиска JPEG-категорий; на CPU используется чистый
PyTorch fallback, но пакет всё равно требуется для импорта модуля).


In [ ]:
# %% Установка зависимостей (эквивалент environment.yml, без активации conda).
import platform
import subprocess
import sys

_PACKAGES = [
    "numpy", "scipy", "pandas", "pillow", "scikit-learn", "pyyaml",
    "python-dotenv", "pyarrow", "cffi", "threadpoolctl", "tensorboard",
    "albumentations", "opencv-python-headless", "timm",
    "torch", "torchvision",
    "triton-windows" if platform.system() == "Windows" else "triton",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *_PACKAGES], check=True)
print("Зависимости установлены:", ", ".join(_PACKAGES))


## 2. Корень проекта, `.env` и проверка данных

Пути к данным и оборудованию берутся из `.env` (см. `.env.example`), приоритет —
`environment > .env > YAML > defaults` (`src/config.py::RuntimeEnvironment`).
Протокол валидации уже зафиксирован и лежит в `runs/validation_protocol_20260908/`
(входит в этот репозиторий) — пересчитывать его не нужно и нельзя: он неизменяемый
манифест ролей train/development/holdout с проверкой чек-сумм.

`DCT_djpeg.pth` (предобученные веса JPEG-ветки) нужен **только для стадии 1**
(`disentangle_b2_li760_r8_long`, обучение с нуля). Стадии 2 и 3 — дотюны от EMA
предыдущего чекпоинта и не трогают pretrained-веса. RGB-энкодер (`pvt_v2_b2_li`)
скачивается автоматически через `timm` при первом запуске стадии 1 (нужен интернет
либо заранее прогретый `torch`/`timm` кэш).


In [ ]:
# %% Импорт numpy раньше torch — на Windows иначе конфликт инициализации MKL
# (тот же порядок используют все ноутбуки этого проекта).
import numpy as np
import os
import shutil
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (ROOT / "src").is_dir():
    raise RuntimeError("Откройте ноутбук из корня репозитория curly-guacamole.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

env_path = ROOT / ".env"
if not env_path.exists() and (ROOT / ".env.example").exists():
    shutil.copy(ROOT / ".env.example", env_path)
    print("Создан .env из .env.example — при необходимости задайте GPU/batch/workers.")

import global_config

data_path = global_config.PathsConfig.current_data_path() if False else (ROOT / global_config.DATA_PATH)
protocol_path = ROOT / "runs" / "validation_protocol_20260908" / "protocol"
jpeg_pretrained = ROOT / "DCT_djpeg.pth"

print("Корень проекта:", ROOT)
print("Данные (AIIJC_DATA_PATH):", data_path, "— найдены:", (data_path / "train_stage1").exists())
print("Протокол валидации:", protocol_path, "— найден:", protocol_path.exists())
print("DCT_djpeg.pth (нужен только для стадии 1):", jpeg_pretrained, "— найден:", jpeg_pretrained.exists())


**TODO (авторам):** если `train_stage1`/`test_stage1` не найдены — распакуйте
данные соревнования в `./data` (или другой путь, указанный в `.env` как
`AIIJC_DATA_PATH`) так, чтобы получилось `data/train_stage1/stage1/train.csv` и
`data/test_stage1/test_stage1/test.csv`, как описано в `AIIJC_RULES.md`. Данные
не входят в репозиторий и не должны копироваться — только пути в `.env`.


## 3. Цепочка конфигов и стадий обучения

Финальный конфиг наследует (`extends`) цепочку из шести родителей — каждое звено
меняет ровно один аспект относительно родителя (архитектурная линия, не гиперпараметр):

```text
configs/baseline.yaml                              # PVT-v2-B2 + native JPEG + LocalFusion + EMCAD, 640, 6 эпох
  → configs/experiments/disentangle_fuse.yaml            # + DG-Force (PFD/EFD) на strides 4/8/16/32, режим fuse
    → configs/experiments/disentangle_fuse_cross32.yaml  # + cross-attention 16→32 на страйде 32
      → configs/experiments/disentangle_b2_li760_long.yaml     # энкодер pvt_v2_b2_li, RGB 760, 18 эпох (3 полных прохода)
        → configs/experiments/disentangle_b2_li760_r8_long.yaml   # disentangle_reduction=8 (бюджет: 99.705 GFLOPs при 760)
          → configs/experiments/disentangle_b2_li760_r8_all_data_ft.yaml         # дотюн 3×3 прохода на train+dev+holdout
            → configs/experiments/disentangle_b2_li760_r8_all_data_hard_pixel_ft.yaml  # + hard-pixel loss (ФИНАЛ)
```

Обучение состоит из **трёх реально запускаемых стадий** (промежуточные конфиги
выше — только наследование архитектуры, отдельно не тренируются):

| Стадия | Конфиг | Эпох | Старт | LR (enc/jpeg/head) | Что нового |
|---|---|---|---|---|---|
| A | `disentangle_b2_li760_r8_long` | 18 (15 sampled + 3 full-pass) | pretrained (RGB timm + `DCT_djpeg.pth`) | 1e-4 / 3e-4 / 3e-4 (baseline) | базовая архитектура на train, валидация на development |
| B | `disentangle_b2_li760_r8_all_data_ft` | 3 (все full-pass) | EMA чекпоинта A (`best.pt`) | 1e-5 / 3e-5 / 3e-5 | `train_all_data=true` — train+development+holdout |
| C (финал) | `disentangle_b2_li760_r8_all_data_hard_pixel_ft` | 3 (все full-pass) | EMA чекпоинта B, снятого после 3 эпох | те же 1e-5 / 3e-5 / 3e-5, `scheduler: none` | `loss.hard_pixel_weight=0.1` (доля 0.1, радиус исключения границы 2px) + Triton backward JPEG-ветки, foreach-нормализация градиентов |

Валидация (подбор чекпоинта/порогов по development) есть только на стадии A.
Стадии B и C — «слепой» дотюн на объединённых данных без валидации: это
осознанный трейд-офф «использовать всю размеченную выборку перед сдачей» за счёт
отсутствия отдельного контроля переобучения на этих двух стадиях — фиксируем это
явно, а не скрываем.

Сиды: `train.seed = 42` (не переопределяется ни в одном звене цепочки) фиксируются
для `random`/`numpy`/`torch` внутри `ExperimentRunner` (`set_random_seed`, см.
`src/training/base.py`) на старте каждой стадии автоматически — отдельно
фиксировать их в ноутбуке не нужно. `torch.use_deterministic_algorithms` не
включён (`deterministic=False` по умолчанию) — это осознанный компромисс в
пользу скорости (`cudnn.benchmark=True`), а не забытый шаг.


In [ ]:
# %% Загружаем финальный конфиг и печатаем разрешённые (после наследования) параметры.
from src.config import load_experiment_config

FINAL_CONFIG_PATH = ROOT / "configs" / "experiments" / "disentangle_b2_li760_r8_all_data_hard_pixel_ft.yaml"
final_cfg = load_experiment_config(FINAL_CONFIG_PATH)

print("run_name:", final_cfg.run_name)
print("seed:", final_cfg.seed)
print("encoder / image_size:", final_cfg.model.encoder, final_cfg.dataset.image_size)
print("disentangle levels/mode/reduction/cross:", final_cfg.model.disentangle_levels,
      final_cfg.model.disentangle_mode, final_cfg.model.disentangle_reduction,
      final_cfg.model.disentangle_cross_strides)
print("epochs / full_pass_epochs:", final_cfg.train.epochs, final_cfg.train.full_pass_epochs)
print("LR enc/jpeg/head:", final_cfg.train.encoder_lr, final_cfg.train.jpeg_lr, final_cfg.train.head_lr)
print("hard_pixel weight/fraction/radius:", final_cfg.loss.hard_pixel_weight,
      final_cfg.loss.hard_pixel_fraction, final_cfg.loss.hard_pixel_radius)
print("devices / batch / accumulation / amp:", final_cfg.train.devices, final_cfg.train.batch_size,
      final_cfg.train.grad_accum_steps, final_cfg.train.amp)


## 4. Бюджет GFLOPs (единый для всех трёх стадий)

Три стадии дообучают одну и ту же архитектуру (меняются только loss/LR/данные,
не модель), поэтому бюджет GFLOPs достаточно посчитать один раз на финальном
конфиге. `count_gflops` считает полный eval-forward через `FlopCounterMode` и
требует явный `native_size` — стоимость JPEG-ветки зависит от исходного
разрешения кадра, а не от `dataset.image_size` (в которое ресайзится RGB-вход).
Референс — native JPEG Full HD (`1080×1920`), как во всех карточках экспериментов
этой линии; более крупные исходники требуют отдельного пересчёта бюджета.


In [ ]:
# %% Проверка бюджета: обязательный gate ПЕРЕД любым обучением, не после.
import torch

from src.budget import count_gflops
from src.training.builders import build_model

NATIVE_SIZE = (1080, 1920)  # Full HD — референсный «худший обычный» исходник линии disentangle_b2_li760_r8_*.

with torch.device("meta"):
    budget_model = build_model(final_cfg.model, aux_weight=final_cfg.loss.aux_weight, pretrained=False).eval()
    gflops = count_gflops(budget_model, final_cfg.dataset.image_size, native_size=NATIVE_SIZE)
del budget_model

print(f"Полный eval-forward: {gflops:.3f} GFLOPs при native {NATIVE_SIZE}, RGB-входе {final_cfg.dataset.image_size}")
assert gflops <= 100, f"{gflops:.2f} GFLOPs превышает лимит соревнования (100 GFLOPs/изображение)"
print("Лимит 100 GFLOPs соблюдён. Латентность ≤50 мс на H100 замеряется отдельно на сервере — не через FlopCounterMode.")


**TODO (авторам):** впишите здесь фактическое число GFLOPs, полученное на вашей
машине (у native-JPEG-ветки операции считаются по-разному на CUDA/CPU из-за
Triton-lookup, см. `configs/experiments/gpu_training_optimizations.md`, поэтому
запускайте эту ячейку на том устройстве, на котором будет считаться latency),
и явно укажите, какой native-размер тестового набора вы проверили дополнительно
(Full HD — не гарантированный максимум, только референс линии экспериментов).


## 5. Стадия A — `disentangle_b2_li760_r8_long` (18 эпох, обучение с нуля)

15 эпох по 24 000 сэмплированных примеров (25% негативов, кроп-аугментации),
затем 3 полных прохода по train на полных кадрах без кропа. DG-Force работает
на strides 4/8/16/32, cross-attention только 16→32 (страйд 8 стоил бы
дополнительных ~15 GFLOPs — сознательно исключён, см. комментарий в
`disentangle_fuse_cross32.yaml`). BiFPN и `parallel_16_32`/`return_to_stride4`
отключены. Резюмируется чекпоинтом `runs/disentangle_b2_li760_r8_long/ckpt/best.pt`
(выбран по development AIC с весом малых масок 1.6) и `ckpt/last.pt`.

Повторный запуск этой ячейки **продолжает** обучение (`resume: true`) с последней
сохранённой эпохи, а не начинает заново — так безопасно прерывать и возобновлять
многочасовой прогон.


In [ ]:
# %% Стадия A: длинное обучение с нуля (18 эпох, RGB 760, DG-Force + cross-attention 16→32).
from src.training.engine import run_experiment

stage_a_cfg = load_experiment_config(ROOT / "configs/experiments/disentangle_b2_li760_r8_long.yaml")
print("Стадия A:", stage_a_cfg.run_name, "| эпох:", stage_a_cfg.train.epochs,
      "| полных проходов:", stage_a_cfg.train.full_pass_epochs)

run_a = run_experiment(stage_a_cfg)
run_a.summary


## 6. Стадия B — `disentangle_b2_li760_r8_all_data_ft` (дотюн на всех данных, 3 полных прохода)

Старт: EMA-веса `runs/disentangle_b2_li760_r8_long/ckpt/best.pt`. Новый optimizer/
scheduler/EMA (дотюн — не resume того же run). `train.train_all_data=true`:
обучение идёт на train **+ development + holdout**, включая проверенные
originals — валидации на этой стадии нет, чекпоинт и пороги уже зафиксированы
на стадии A. LR понижен на порядок (1e-5 / 3e-5 / 3e-5), cosine-затухание до 2%
исходного LR за 3 полных прохода.


In [ ]:
# %% Стадия B: финальный дотюн на train+development+holdout (без валидации).
stage_b_cfg = load_experiment_config(ROOT / "configs/experiments/disentangle_b2_li760_r8_all_data_ft.yaml")
print("Стадия B:", stage_b_cfg.run_name, "| finetune_from:", stage_b_cfg.train.finetune_from,
      "| train_all_data:", stage_b_cfg.train.train_all_data)

source_ckpt = stage_b_cfg.paths.runs_path / stage_b_cfg.train.finetune_from
if not source_ckpt.is_file():
    raise FileNotFoundError(f"Нет чекпоинта стадии A: {source_ckpt}. Сначала выполните ячейку стадии A.")

run_b = run_experiment(stage_b_cfg)
run_b.summary


### Снапшот стадии B перед стадией C

Финальный конфиг ссылается на снимок стадии B `disentangle_b2_li760_r8_all_data_ft_ep3`
(а не на `disentangle_b2_li760_r8_all_data_ft` напрямую) — так делается **копия**
завершённого чекпоинта под отдельным именем, прежде чем продолжать обучение дальше.
Это защищает от ситуации, когда стадия C (или любой другой альтернативный дотюн от
той же точки) случайно `resume`-нется в каталог стадии B и испортит его как
самостоятельную, воспроизводимую промежуточную точку. Копия делается один раз —
после того как стадия B выше уже отработала все 3/3 эпохи.


In [ ]:
# %% Копия чекпоинта стадии B под именем, которое ждёт финальный конфиг (once).
stage_b_dir = ROOT / "runs" / "disentangle_b2_li760_r8_all_data_ft"
stage_b_snapshot_dir = ROOT / "runs" / "disentangle_b2_li760_r8_all_data_ft_ep3"
final_ckpt_dir = ROOT / "runs" / final_cfg.run_name / "ckpt"

if (final_ckpt_dir / "last.pt").exists():
    print("У финального run уже есть свой ckpt/last.pt — снапшот стадии B больше не нужен.")
elif stage_b_snapshot_dir.exists():
    print("Снапшот уже существует:", stage_b_snapshot_dir)
else:
    if not (stage_b_dir / "ckpt" / "last.pt").exists():
        raise FileNotFoundError(f"Стадия B ещё не завершена: нет {stage_b_dir / 'ckpt' / 'last.pt'}")
    shutil.copytree(stage_b_dir, stage_b_snapshot_dir)
    print("Скопировано:", stage_b_dir, "->", stage_b_snapshot_dir)


## 7. Стадия C (финал) — `disentangle_b2_li760_r8_all_data_hard_pixel_ft`

Старт: EMA `runs/disentangle_b2_li760_r8_all_data_ft_ep3/ckpt/last.pt`. Новый
optimizer/EMA. Ещё 3 полных прохода по train+development+holdout, те же LR, но
**постоянные** (`scheduler: none`, без warmup) — на этой стадии расписание LR
сознательно не перенастраивалось повторно, чтобы изолировать эффект
hard-pixel-лосса от эффекта нового LR-расписания (см. `notebooks/ablations.ipynb`
про то, почему это разделение факторов важно).

Loss = исходный (BCE+Dice+aux+DG-Force patch/edge) `+ 0.1 * (hard_positive_BCE +
hard_negative_BCE)`, где для каждого изображения отдельно берутся худшие 10%
допустимых пикселей внутри GT и вне GT (полоса ±2px вокруг границы исключена из
допустимой области — там супервизия остаётся обычной, не «жёсткой»). Пустая
допустимая область даёт нулевой вклад; для чистых (негативных) изображений
работает только hard-negative часть. Валидации на этой стадии нет — это
финальные веса.

Дополнительно включены две GPU-оптимизации, не меняющие архитектуру/loss/LR
(`configs/experiments/gpu_training_optimizations.md`): тайловый Triton backward
для градиентов JPEG-весов вместо full-resolution one-hot тензора, и
foreach-нормализация накопленных градиентов по группам device/dtype. Обе влияют
только на скорость и порядок редукции чисел (возможны float-отличия), не на
семантику обучения.


In [ ]:
# %% Стадия C (ФИНАЛ): + hard-pixel loss, Triton backward, foreach-нормализация.
final_run = run_experiment(final_cfg)
final_run.summary


**TODO (авторам):** после реального прогона впишите здесь фактические метрики
из `runs/disentangle_b2_li760_r8_all_data_hard_pixel_ft/summary.json` (или из
`final_run.summary` выше) — combined AIC, Dice_pos, FPR_neg и выбранные пороги.
Так как стадии B/C обучаются без валидации, эти метрики фактически наследуются
от стадии A (последней стадии с development-оценкой) и служат ожиданием, а не
гарантией: реальное качество финальных весов проверяется только holdout-оценкой
(раздел 9) и итоговым лидербордом.


## 8. Инференс и сборка submission

Стадии B и C — «слепой» дотюн (`train_all_data=true`): у финального run **нет
своего** `summary.json['best']` (валидация на нём не запускалась, см.
`tests/test_blind_finetune.py::test_blind_epochs_never_validate_and_resume`), а
значит `create_submission(..., thresholds=None)` не сможет сам найти пороги и
упадёт с `ValueError`. Поэтому пороги (`mask_threshold`, `cls_threshold`,
`min_area`) берём из **стадии A** — последнего run, где реально была валидация
и подбор операционной точки по development AIC (вес малых масок 1.6) — и
передаём их явно. Веса при этом берём финального чекпоинта (`ckpt/last.pt`,
EMA приоритетнее raw), пороги — только с чекпоинта стадии A.


In [ ]:
# %% Пороги берём с той стадии, где была валидация (A), веса — с финальной (C).
import json

from src.inference.predict import ThresholdConfig
from src.inference.submission import create_submission

stage_a_summary = json.loads((ROOT / "runs" / stage_a_cfg.run_name / "summary.json").read_text(encoding="utf-8"))
best = stage_a_summary["best"]
thresholds = ThresholdConfig(
    mask_threshold=float(best["mask_threshold"]),
    cls_threshold=float(best["cls_threshold"]),
    min_area=float(best["min_area"]),
    area_cap=float(best.get("area_cap", 0.0)),
    n_bins=stage_a_cfg.eval.n_bins,  # не переопределяется ни в одном звене цепочки (по умолчанию 256).
)
print("Пороги (со стадии A):", thresholds)

run_dir = ROOT / "runs" / final_cfg.run_name
output_dir = ROOT / "submissions" / final_cfg.run_name

csv_path = create_submission(run_dir, output_dir, thresholds=thresholds, checkpoint_name="last.pt")
print("submission.csv:", csv_path)
print("Маски:", output_dir / "predictions")


In [ ]:
# %% Упаковка в ZIP: submission.csv и predictions/ должны лежать в КОРНЕ архива
# (без вложенной папки) — требование AIIJC_RULES.md.
import shutil

archive_path = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir)
print("Готовый архив для отправки:", archive_path)


## 9. (Опционально) Независимый holdout

Отдельная, одноразовая финальная проверка на holdout-срезе протокола с
**замороженными** порогами — **не подбор**, а контроль. После вызова
создаётся `holdout_claim.json`, и этот run больше нельзя ни продолжать обучать,
ни дотюнить дальше. Поэтому эта ячейка закомментирована и не выполняется
автоматически при полном прогоне ноутбука — запускайте её осознанно, отдельно,
когда финальный чекпоинт уже точно готов к сдаче:

```bash
python -m src.eval --run runs/disentangle_b2_li760_r8_all_data_hard_pixel_ft
```

## Ссылки

- Полная инструкция и таблица ожидаемых чисел — `README.md` в корне репозитория.
- Почему именно такая архитектура и loss — `notebooks/eda.ipynb`, `notebooks/model_comparison.ipynb`, `notebooks/ablations.ipynb`.
- Правила соревнования — `AIIJC_RULES.md`.
